# 05. TF Embedding Residual Ablation

`04` 임베딩 실험이 baseline보다 악화된 원인을 좁히고, residual target 기반 개선 후보를 smoke run으로 비교합니다.

- E00: `04` 구조 재현
- E01: residual target only
- E02/E03: residual + lower learning rate
- E04: residual + smaller embeddings
- E05: residual + rare complex bucket
- E06: residual + no complex_id
- 기본값: `RUN_MODE = "smoke"`

In [ ]:
from pathlib import Path
import json
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)
print("python", sys.version)
print("tensorflow", tf.__version__)
print("pandas", pd.__version__)

In [ ]:
# 1) 경로와 실행 설정
current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

DATA_PATH = PROJECT_DIR / "data" / "processed" / "transactions.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"  # "smoke" or "full"
RANDOM_STATE = 42
SMOKE_LIMITS = {"train": 200_000, "valid": 50_000, "test": 50_000, "recent_holdout": 50_000}

BATCH_SIZE = 8192
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 4

assert RUN_MODE in {"smoke", "full"}
assert DATA_PATH.exists(), DATA_PATH
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
print(PROJECT_DIR, RUN_MODE)

In [ ]:
# 2) 데이터 로드와 Policy B 필터링
USECOLS = [
    "transaction_id", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "floor", "age_years",
    "deal_date", "trade_type", "is_cancelled", "price_total", "price_per_m2",
    "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days",
]
DTYPES = {
    "transaction_id": "string", "complex_id": "string", "legal_dong_code": "string", "sgg_code": "string",
    "area_m2": "float32", "floor": "float32", "age_years": "float32", "trade_type": "string",
    "is_cancelled": "Int8", "price_total": "float32", "price_per_m2": "float32",
    "complex_prev_price_per_m2": "float32", "complex_prev_missing": "Int8", "prev_deal_gap_days": "float32",
}
raw_df = pd.read_csv(DATA_PATH, usecols=USECOLS, dtype=DTYPES, parse_dates=["deal_date"])
raw_df["trade_type"] = raw_df["trade_type"].fillna("unknown")
df = raw_df.loc[(raw_df["is_cancelled"] == 0) & raw_df["trade_type"].isin(["중개거래", "unknown"])].copy()
print("rows", len(raw_df), "policy_b", len(df))

In [ ]:
# 3) Feature 생성과 leakage 방지
NUMERIC_FEATURES = [
    "area_m2", "floor", "is_basement_floor", "age_years",
    "log_complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_months",
]
ALL_EMBEDDING_FEATURES = ["complex_id", "legal_dong_code", "sgg_code", "prev_deal_gap_bucket"]
ORIGINAL_EMBEDDING_DIMS = {"complex_id": 32, "legal_dong_code": 16, "sgg_code": 8, "prev_deal_gap_bucket": 3}
SMALL_EMBEDDING_DIMS = {"complex_id": 16, "legal_dong_code": 8, "sgg_code": 4, "prev_deal_gap_bucket": 3}

HARD_LEAKAGE_COLUMNS = {"target", "price_total", "price_per_m2", "deal_date", "transaction_id", "trade_type", "is_cancelled", "complex_prev_price_per_m2", "prev_deal_gap_days"}
assert not ((set(NUMERIC_FEATURES) | set(ALL_EMBEDDING_FEATURES)) & HARD_LEAKAGE_COLUMNS)

def add_features(input_df):
    out = input_df.copy()
    out["target"] = np.log(out["price_per_m2"].astype("float64"))
    out["is_basement_floor"] = (out["floor"] < 0).astype("float32")
    prev_price = out["complex_prev_price_per_m2"].astype("float64")
    out["log_complex_prev_price_per_m2"] = np.where(prev_price > 0, np.log(prev_price), np.nan)
    out["complex_prev_missing"] = out["complex_prev_missing"].fillna(1).astype("float32")
    out["prev_deal_gap_months"] = out["prev_deal_gap_days"].astype("float64") / 30.4375
    gap = out["prev_deal_gap_days"]
    bucket = pd.Series("missing", index=out.index, dtype="string")
    bucket[(gap >= 0) & (gap <= 30)] = "0-30"
    bucket[(gap >= 31) & (gap <= 90)] = "31-90"
    bucket[(gap >= 91) & (gap <= 180)] = "91-180"
    bucket[(gap >= 181) & (gap <= 365)] = "181-365"
    bucket[gap >= 366] = "366+"
    out["prev_deal_gap_bucket"] = bucket.fillna("missing")
    for feature in ALL_EMBEDDING_FEATURES:
        out[feature] = out[feature].fillna("missing").astype("string")
    return out

df = add_features(df)
assert df["target"].notna().all()
assert np.isfinite(df["target"]).all()

In [ ]:
# 4) 시간 기준 split과 smoke sampling
SPLIT_ORDER = ["train", "valid", "test", "recent_holdout"]

def split_frames(policy_df):
    splits = {
        "train": policy_df.loc[policy_df["deal_date"] <= "2023-12-31"],
        "valid": policy_df.loc[(policy_df["deal_date"] >= "2024-01-01") & (policy_df["deal_date"] <= "2024-12-31")],
        "test": policy_df.loc[(policy_df["deal_date"] >= "2025-01-01") & (policy_df["deal_date"] <= "2025-12-31")],
        "recent_holdout": policy_df.loc[policy_df["deal_date"] >= "2026-01-01"],
    }
    for name, frame in splits.items():
        assert len(frame) > 0, name
    return splits

def apply_smoke_sampling(splits):
    if RUN_MODE != "smoke":
        return {k: v.copy() for k, v in splits.items()}
    out = {}
    for name, frame in splits.items():
        limit = SMOKE_LIMITS[name]
        out[name] = frame.sample(n=limit, random_state=RANDOM_STATE).sort_values("deal_date") if len(frame) > limit else frame.copy()
    return out

full_splits = split_frames(df)
run_splits = apply_smoke_sampling(full_splits)
counts_df = pd.DataFrame([{"split": s, "full_rows": len(full_splits[s]), "run_rows": len(run_splits[s])} for s in SPLIT_ORDER])
display(counts_df)

In [ ]:
# 5) 실험 설정
EXPERIMENTS = [
    {"experiment_name": "E00_original_04_reproduction", "target_mode": "direct", "learning_rate": 0.001, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 0, "dense_units": [128, 64]},
    {"experiment_name": "E01_residual_target_only", "target_mode": "residual", "learning_rate": 0.001, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 0, "dense_units": [128, 64]},
    {"experiment_name": "E02_residual_lr_0005", "target_mode": "residual", "learning_rate": 0.0005, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 0, "dense_units": [128, 64]},
    {"experiment_name": "E03_residual_lr_0003", "target_mode": "residual", "learning_rate": 0.0003, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 0, "dense_units": [128, 64]},
    {"experiment_name": "E04_residual_small_embedding", "target_mode": "residual", "learning_rate": 0.001, "embedding_dims": SMALL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 0, "dense_units": [128, 64]},
    {"experiment_name": "E05_residual_rare_complex_min5", "target_mode": "residual", "learning_rate": 0.001, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": True, "rare_min_count": 5, "dense_units": [128, 64]},
    {"experiment_name": "E06_residual_no_complex_id", "target_mode": "residual", "learning_rate": 0.001, "embedding_dims": ORIGINAL_EMBEDDING_DIMS, "use_complex_id": False, "rare_min_count": 0, "dense_units": [128, 64]},
]

numeric_medians = run_splits["train"][NUMERIC_FEATURES].median(numeric_only=True).astype("float32")

def embedding_features(config):
    return ALL_EMBEDDING_FEATURES if config["use_complex_id"] else [f for f in ALL_EMBEDDING_FEATURES if f != "complex_id"]

def prepare_split_for_config(split_df, train_complex_counts, config):
    out = split_df.copy()
    if config["rare_min_count"] > 0 and "complex_id" in embedding_features(config):
        rare_ids = set(train_complex_counts[train_complex_counts < config["rare_min_count"]].index.astype(str))
        values = out["complex_id"].astype("string").astype(str)
        out["complex_id"] = values.mask(values.isin(rare_ids), "rare_complex").astype("string")
    return out

def base_log(split_df):
    return split_df["log_complex_prev_price_per_m2"].fillna(numeric_medians["log_complex_prev_price_per_m2"]).to_numpy(dtype="float32")

def make_inputs(split_df, features):
    numeric_df = split_df[NUMERIC_FEATURES].copy().fillna(numeric_medians)
    inputs = {"numeric_input": numeric_df.to_numpy(dtype="float32")}
    for feature in features:
        values = np.asarray(split_df[feature].fillna("missing").astype("string").astype(str).tolist(), dtype=str).reshape(-1, 1)
        inputs[f"{feature}_input"] = tf.convert_to_tensor(values, dtype=tf.string)
    return inputs

def y_for(split_df, target_mode):
    target = split_df["target"].to_numpy(dtype="float32")
    return target if target_mode == "direct" else target - base_log(split_df)

def final_log_pred(split_df, raw_pred, target_mode):
    raw_pred = np.asarray(raw_pred, dtype="float64").reshape(-1)
    return raw_pred if target_mode == "direct" else base_log(split_df).astype("float64") + raw_pred

In [ ]:
# 6) 모델, 평가 함수
train_complex_counts = run_splits["train"]["complex_id"].astype("string").astype(str).value_counts()

def build_preprocessors(config, prepared_train):
    features = embedding_features(config)
    train_inputs = make_inputs(prepared_train, features)
    normalizer = keras.layers.Normalization(name="numeric_normalization")
    normalizer.adapt(train_inputs["numeric_input"])
    lookups = {}
    for feature in features:
        lookup = keras.layers.StringLookup(num_oov_indices=1, mask_token=None, name=f"{feature}_lookup")
        lookup.adapt(train_inputs[f"{feature}_input"])
        lookups[feature] = lookup
    return features, train_inputs, normalizer, lookups

def build_model(config, features, normalizer, lookups, seed_offset=0):
    tf.keras.utils.set_random_seed(RANDOM_STATE + seed_offset)
    numeric_input = keras.Input(shape=(len(NUMERIC_FEATURES),), name="numeric_input", dtype="float32")
    parts = [normalizer(numeric_input)]
    inputs = [numeric_input]
    for feature in features:
        inp = keras.Input(shape=(1,), name=f"{feature}_input", dtype=tf.string)
        idx = lookups[feature](inp)
        dim = config["embedding_dims"][feature]
        emb = keras.layers.Embedding(lookups[feature].vocabulary_size(), dim, name=f"{feature}_embedding")(idx)
        inputs.append(inp)
        parts.append(keras.layers.Flatten(name=f"{feature}_flatten")(emb))
    x = keras.layers.Concatenate(name="feature_concat")(parts)
    for unit in config["dense_units"]:
        x = keras.layers.Dense(unit, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-5))(x)
        if unit >= 128:
            x = keras.layers.Dropout(0.10)(x)
        else:
            x = keras.layers.Dropout(0.05)(x)
    out = keras.layers.Dense(1)(x)
    model = keras.Model(inputs=inputs, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

def evaluate(split_df, pred_log, config, split_name):
    y_true = split_df["target"].to_numpy(dtype="float64")
    pred_log = np.asarray(pred_log, dtype="float64").reshape(-1)
    pred_ppm = np.exp(pred_log)
    actual_ppm = split_df["price_per_m2"].to_numpy(dtype="float64")
    pred_total = pred_ppm * split_df["area_m2"].to_numpy(dtype="float64")
    actual_total = split_df["price_total"].to_numpy(dtype="float64")
    mape_mask = actual_ppm > 0
    return {
        "run_mode": RUN_MODE,
        "experiment_name": config["experiment_name"],
        "target_mode": config["target_mode"],
        "learning_rate": config["learning_rate"],
        "use_complex_id": config["use_complex_id"],
        "rare_min_count": config["rare_min_count"],
        "embedding_dims": json.dumps(config["embedding_dims"], ensure_ascii=False),
        "dense_units": json.dumps(config["dense_units"]),
        "split": split_name,
        "rows": len(split_df),
        "log_mae": float(mean_absolute_error(y_true, pred_log)),
        "log_rmse": float(math.sqrt(mean_squared_error(y_true, pred_log))),
        "price_per_m2_mae": float(mean_absolute_error(actual_ppm, pred_ppm)),
        "price_per_m2_mape": float(np.mean(np.abs((actual_ppm[mape_mask] - pred_ppm[mape_mask]) / actual_ppm[mape_mask]))),
        "total_price_mae_manwon": float(mean_absolute_error(actual_total, pred_total)),
    }

In [ ]:
# 7) 실험 실행
metrics_rows = []
history_rows = []
prediction_samples = []
training_rows = []

for idx, config in enumerate(EXPERIMENTS):
    tf.keras.backend.clear_session()
    name = config["experiment_name"]
    print("\n===", name, "===")
    prepared = {s: prepare_split_for_config(run_splits[s], train_complex_counts, config) for s in SPLIT_ORDER}
    features, train_inputs, normalizer, lookups = build_preprocessors(config, prepared["train"])
    model = build_model(config, features, normalizer, lookups, seed_offset=idx)
    valid_inputs = make_inputs(prepared["valid"], features)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
    ]
    start = time.perf_counter()
    history = model.fit(
        train_inputs,
        y_for(prepared["train"], config["target_mode"]),
        validation_data=(valid_inputs, y_for(prepared["valid"], config["target_mode"])),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    duration = time.perf_counter() - start
    hdf = pd.DataFrame(history.history)
    hdf.insert(0, "epoch", np.arange(1, len(hdf) + 1))
    hdf.insert(0, "experiment_name", name)
    history_rows.append(hdf)
    training_rows.append({"experiment_name": name, "epochs_ran": len(hdf), "best_epoch": int(hdf["val_loss"].idxmin()) + 1, "best_val_loss": float(hdf["val_loss"].min()), "duration_seconds": duration})

    for split_name in SPLIT_ORDER:
        inputs = make_inputs(prepared[split_name], features)
        raw_pred = model.predict(inputs, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
        pred_log = final_log_pred(prepared[split_name], raw_pred, config["target_mode"])
        metrics_rows.append(evaluate(prepared[split_name], pred_log, config, split_name))
        if split_name in {"valid", "test", "recent_holdout"}:
            sample = prepared[split_name][["transaction_id", "deal_date", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "price_total", "price_per_m2", "target"]].copy()
            sample["experiment_name"] = name
            sample["split"] = split_name
            sample["pred_target"] = pred_log
            sample["pred_price_per_m2"] = np.exp(pred_log)
            if len(sample) > 50:
                sample = sample.sample(n=50, random_state=RANDOM_STATE)
            prediction_samples.append(sample)

metrics_df = pd.DataFrame(metrics_rows)
history_df = pd.concat(history_rows, ignore_index=True)
training_df = pd.DataFrame(training_rows)
predictions_sample_df = pd.concat(prediction_samples, ignore_index=True)
display(metrics_df)

In [ ]:
# 8) baseline/04/E01 비교와 저장
BASELINE_PATH = OUTPUT_DIR / "baseline_regression_metrics.csv"
FOUR_PATH = OUTPUT_DIR / "tf_multi_input_embedding_metrics.csv"
METRICS_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_history.csv"
TRAINING_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_training.csv"
SUMMARY_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_summary.md"
PRED_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_best_predictions_sample.csv"
CONFIG_PATH = OUTPUT_DIR / "tf_embedding_residual_ablation_experiments.json"

baseline_df = pd.read_csv(BASELINE_PATH) if BASELINE_PATH.exists() else pd.DataFrame()
four_df = pd.read_csv(FOUR_PATH) if FOUR_PATH.exists() else pd.DataFrame()
base_candidates = baseline_df.loc[
    (baseline_df.get("policy") == "policy_b_broker_unknown")
    & (baseline_df.get("split") == "valid")
    & (baseline_df.get("model").isin(["linear_regression", "ridge_alpha_1"]))
].copy() if len(baseline_df) else pd.DataFrame()
base_best = base_candidates.sort_values("log_mae").iloc[0].to_dict() if len(base_candidates) else {}
base_model = base_best.get("model")
base_run_mode = base_best.get("run_mode")
base_subset = baseline_df[(baseline_df["policy"] == "policy_b_broker_unknown") & (baseline_df["model"] == base_model)] if len(baseline_df) and base_model else pd.DataFrame()

def lookup_metric(df, split, col="log_mae"):
    if len(df) == 0:
        return np.nan
    rows = df.loc[df["split"] == split]
    return float(rows.iloc[0][col]) if len(rows) else np.nan

comp = metrics_df.copy()
comp["baseline_model"] = base_model
comp["baseline_run_mode"] = base_run_mode
comp["baseline_log_mae"] = comp["split"].map(lambda s: lookup_metric(base_subset, s))
comp["four_log_mae"] = comp["split"].map(lambda s: lookup_metric(four_df, s))
e01_subset = comp.loc[comp["experiment_name"] == "E01_residual_target_only", ["split", "log_mae"]].set_index("split")["log_mae"].to_dict()
comp["e01_log_mae"] = comp["split"].map(lambda s: e01_subset.get(s, np.nan))
comp["delta_vs_baseline"] = comp["log_mae"] - comp["baseline_log_mae"]
comp["delta_vs_04"] = comp["log_mae"] - comp["four_log_mae"]
comp["delta_vs_e01"] = comp["log_mae"] - comp["e01_log_mae"]
comp["beats_baseline"] = comp["delta_vs_baseline"] < -1e-9
comp["beats_04"] = comp["delta_vs_04"] < -1e-9
comp["beats_e01"] = comp["delta_vs_e01"] < -1e-9
metrics_df = comp

best_valid = metrics_df.loc[metrics_df["split"] == "valid"].sort_values("log_mae").iloc[0]
best_name = best_valid["experiment_name"]
best_preds = predictions_sample_df.loc[predictions_sample_df["experiment_name"] == best_name].copy()

metrics_df.to_csv(METRICS_PATH, index=False)
history_df.to_csv(HISTORY_PATH, index=False)
training_df.to_csv(TRAINING_PATH, index=False)
best_preds.to_csv(PRED_PATH, index=False)
CONFIG_PATH.write_text(json.dumps(EXPERIMENTS, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

def md_table(df, floatfmt=".6f"):
    x = df.copy()
    for col in x.select_dtypes(include=["float", "float32", "float64"]).columns:
        x[col] = x[col].map(lambda v: format(v, floatfmt) if pd.notna(v) else "")
    x = x.astype("string").fillna("")
    lines = ["| " + " | ".join(x.columns) + " |", "| " + " | ".join(["---"] * len(x.columns)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in x.values.tolist()]
    return "\n".join(lines)

valid_cols = ["experiment_name", "target_mode", "learning_rate", "use_complex_id", "rare_min_count", "log_mae", "price_per_m2_mape", "baseline_log_mae", "four_log_mae", "e01_log_mae", "delta_vs_baseline", "delta_vs_04", "delta_vs_e01", "beats_baseline", "beats_04", "beats_e01"]
valid_comp = metrics_df.loc[metrics_df["split"] == "valid", valid_cols].sort_values("log_mae")
all_comp = metrics_df[["experiment_name", "split", "log_mae", "price_per_m2_mape", "baseline_log_mae", "four_log_mae", "e01_log_mae", "delta_vs_baseline", "delta_vs_04", "delta_vs_e01", "beats_baseline", "beats_04", "beats_e01"]]

summary = []
summary.append("# TF Embedding Residual Ablation 결과")
summary.append("")
summary.append("## 1. 목적")
summary.append("04 임베딩 모델 악화 원인을 좁히고 residual target 기반 튜닝 후보를 smoke run으로 비교했다.")
summary.append("")
summary.append("## 2. 실험 목록")
summary.append(md_table(pd.DataFrame(EXPERIMENTS)[["experiment_name", "target_mode", "learning_rate", "use_complex_id", "rare_min_count", "dense_units"]]))
summary.append("")
summary.append("## 3. Split")
summary.append(md_table(counts_df, floatfmt=".0f"))
summary.append("")
summary.append("## 4. Valid 순위")
summary.append(md_table(valid_comp))
summary.append("")
summary.append("## 5. 전체 metrics")
summary.append(md_table(all_comp))
summary.append("")
summary.append("## 6. 1차 판단")
summary.append(f"valid 기준 best experiment는 `{best_name}`이고 valid log_mae는 `{best_valid['log_mae']:.6f}`이다.")
summary.append(f"03 full linear baseline 대비 delta는 `{best_valid['delta_vs_baseline']:.6f}`이다.")
summary.append(f"04 direct embedding 대비 delta는 `{best_valid['delta_vs_04']:.6f}`이다.")
summary.append(f"E01 residual only 대비 delta는 `{best_valid['delta_vs_e01']:.6f}`이다.")
if bool(best_valid["beats_baseline"]):
    summary.append("valid 기준 03 full linear baseline을 이겼다.")
else:
    summary.append("valid 기준 03 full linear baseline은 아직 이기지 못했다.")
if bool(best_valid["beats_e01"]):
    summary.append("E01 residual only보다 추가 튜닝 효과가 있었다.")
else:
    summary.append("E01 residual only보다 명확한 추가 튜닝 이득은 없었다.")
summary.append("")
summary.append("## 7. 생성 산출물")
summary.append(f"- metrics: `{METRICS_PATH}`")
summary.append(f"- history: `{HISTORY_PATH}`")
summary.append(f"- training: `{TRAINING_PATH}`")
summary.append(f"- predictions: `{PRED_PATH}`")
summary.append(f"- experiments: `{CONFIG_PATH}`")
SUMMARY_PATH.write_text("\n".join(summary), encoding="utf-8")

print("best", best_name, float(best_valid["log_mae"]))
print(METRICS_PATH)
print(SUMMARY_PATH)